# ⚡ InklusiKerja — Step 4: FastAPI Endpoint Demo

Notebook ini mendemonstrasikan logika endpoint API tanpa menjalankan server.

**Prerequisite:** Notebook 01–03 sudah dijalankan.

**Endpoint asli** (jalankan via terminal):
```bash
uvicorn 04_api:app --host 0.0.0.0 --port 8000 --reload
```

**Yang dilakukan di notebook ini:**
- Simulasi request `POST /api/match` (FAISS-based)
- Simulasi request `POST /api/match/direct` (direct scoring)
- Contoh payload & response

## 1. Setup — Load Engine

In [ ]:
import os, sys, importlib.util
import numpy as np
import json

# Load recommendation engine dari 03_recommendation_engine.py
spec = importlib.util.spec_from_file_location(
    "recommendation_engine", "03_recommendation_engine.py")
rec_mod = importlib.util.module_from_spec(spec)
sys.modules["recommendation_engine"] = rec_mod
spec.loader.exec_module(rec_mod)

RecommendationEngine = rec_mod.RecommendationEngine
KandidatProfile      = rec_mod.KandidatProfile
normalize_skill_name = rec_mod.normalize_skill_name
get_accommodation    = rec_mod.get_accommodation

engine = RecommendationEngine(index_dir="data/index", model_key="multilingual_minilm")
print("✅ Engine siap")

## 2. Simulasi POST /api/match (FAISS Recommendation)

In [ ]:
# Payload contoh (mirip JSON request ke API)
match_request = {
    "disability_type"   : "Gangguan Pendengaran (Tunarungu)",
    "skills"            : ["Python", "SQL", "Tableau", "Power BI"],
    "functional_profile": "Data analyst berpengalaman 2 tahun. Mengalami gangguan pendengaran sejak kecil, bekerja efektif via komunikasi tertulis.",
    "preferred_level"   : "Mid level",
    "top_k"             : 5,
    "weights"           : {"semantic": 0.35, "skill": 0.05, "disability": 0.60},
    "min_semantic"      : 0.43,
}

kandidat = KandidatProfile(
    disability_type    = match_request["disability_type"],
    skills             = match_request["skills"],
    functional_profile = match_request["functional_profile"],
    preferred_level    = match_request["preferred_level"],
)

results = engine.recommend(
    kandidat,
    top_k        = match_request["top_k"],
    weights      = match_request["weights"],
    min_semantic = match_request["min_semantic"],
)

response = {
    "status"         : "success",
    "total_results"  : len(results),
    "kandidat_query" : kandidat.to_query_text(),
    "weights_used"   : match_request["weights"],
    "recommendations": [
        {
            "rank"                     : r.rank,
            "job_id"                   : r.job_id,
            "job_title"                : r.job_title,
            "level"                    : r.level,
            "semantic_score"           : r.semantic_score,
            "skill_match_score"        : r.skill_match_score,
            "disability_match_score"   : r.disability_match_score,
            "final_score"              : r.final_score,
            "matched_skills"           : r.matched_skills,
            "skill_gap"                : r.skill_gap,
            "accommodation_suggestions": r.accommodation_suggestions,
            "explanation"              : r.explanation,
        } for r in results
    ]
}

print(json.dumps(response, ensure_ascii=False, indent=2)[:2000], "...")

## 3. Simulasi POST /api/match/direct (Direct Scoring)

In [ ]:
def generate_direct_explanation(semantic_score, skill_ratio, matched_skills, skill_gap, has_skills):
    parts = []
    if semantic_score >= 0.80: parts.append("Deskripsi profilmu sangat relevan dengan deskripsi pekerjaan ini.")
    elif semantic_score >= 0.60: parts.append("Deskripsi profilmu cukup relevan dengan pekerjaan ini.")
    elif semantic_score >= 0.40: parts.append("Ada kesamaan antara profil kamu dan pekerjaan ini, tapi tidak terlalu kuat.")
    else: parts.append("Profil kamu kurang relevan dengan deskripsi pekerjaan ini.")
    if has_skills:
        if skill_ratio >= 0.80 and matched_skills:
            parts.append(f"Skill kamu sangat cocok ({round(skill_ratio*100)}%): {', '.join(matched_skills[:3])}.")
        elif skill_ratio >= 0.50 and matched_skills:
            parts.append(f"Skill kamu cocok sebagian ({round(skill_ratio*100)}%): {', '.join(matched_skills[:3])}.")
        elif skill_ratio > 0 and matched_skills:
            parts.append(f"Hanya {round(skill_ratio*100)}% skill cocok: {', '.join(matched_skills[:2])}.")
        else:
            parts.append("Tidak ada skill kamu yang cocok dengan persyaratan.")
        if skill_gap:
            parts.append(f"Skill yang perlu diperkuat: {', '.join(skill_gap[:3])}.")
    else:
        parts.append("Tidak ada daftar skill spesifik — kecocokan dinilai dari relevansi deskripsi.")
    return " ".join(parts)


# Payload direct match
direct_request = {
    "disability_type"    : "Tunanetra",
    "skills"             : ["Python", "SQL", "Tableau"],
    "functional_profile" : "Seorang data analyst berpengalaman menggunakan screen reader JAWS.",
    "job_title"          : "Data Analyst",
    "job_description"    : "Kami mencari data analyst yang mahir SQL dan Python untuk mengolah data bisnis.",
    "job_required_skills": ["Python", "SQL", "Tableau", "Power BI"],
}

# 1. Skill score
candidate_skills_norm = {normalize_skill_name(s) for s in direct_request["skills"] if s.strip()}
job_skills_norm = {normalize_skill_name(s) for s in direct_request["job_required_skills"] if s.strip()}

exact_matched = candidate_skills_norm & job_skills_norm
partial_matched = set()
for cs in candidate_skills_norm:
    for js in job_skills_norm:
        if js not in exact_matched and (cs in js or js in cs) and len(cs) >= 3:
            partial_matched.add(js)

all_matched = exact_matched | partial_matched
skill_gap = list(job_skills_norm - all_matched)
skill_ratio = float(np.clip(
    (len(exact_matched) + 0.6 * len(partial_matched)) / len(job_skills_norm), 0, 1
)) if job_skills_norm else 0.0
matched_skills = list(all_matched)

# 2. Semantic score
profile_text = direct_request["functional_profile"]
job_text = f"{direct_request['job_title']}. {direct_request['job_description']}"

vecs = engine.model.encode(
    [profile_text, job_text],
    normalize_embeddings=True, convert_to_numpy=True
).astype(np.float32)

raw_cosine    = float(np.dot(vecs[0], vecs[1]))
semantic_score = float(np.clip(raw_cosine / 0.9, 0.0, 1.0))

# 3. Final score
final_score = (0.50 * skill_ratio + 0.50 * semantic_score) * 100

dis_score = engine._compute_disability_match(
    direct_request["disability_type"], direct_request["disability_type"])

explanation = generate_direct_explanation(
    semantic_score, skill_ratio, matched_skills, skill_gap,
    bool(direct_request["job_required_skills"])
)

direct_response = {
    "status"         : "success",
    "kandidat_query" : profile_text,
    "job_query"      : job_text,
    "result": {
        "semantic_score"           : round(semantic_score * 100, 1),
        "skill_match_score"        : round(skill_ratio * 100, 1),
        "disability_match_score"   : round(dis_score * 100, 1),
        "final_score"              : round(final_score, 1),
        "matched_skills"           : matched_skills[:5],
        "skill_gap"                : skill_gap[:5],
        "accommodation_suggestions": get_accommodation(direct_request["disability_type"]),
        "explanation"              : explanation,
        "source"                   : "ml_direct",
    }
}

print(json.dumps(direct_response, ensure_ascii=False, indent=2))

## 4. Health Check Simulasi

In [ ]:
health = {
    "status"            : "ok",
    "engine"            : "loaded",
    "total_jobs_indexed": engine.index.ntotal,
    "model"             : engine.config["model_name"],
}
print(json.dumps(health, indent=2))
print("\n✅ Semua endpoint berhasil disimulasikan!")
print("\n▶️  Untuk menjalankan API server sesungguhnya:")
print("   uvicorn 04_api:app --host 0.0.0.0 --port 8000 --reload")